# Part 4 - Gradient Descent in Code (Python, SciPy and Matplotlib)

This converts **Part 3 - Gradient Descent Manual Calculation** (the same problem worked by hand, `docs/Part3_Gradient_Descent_Manual_Calculation_Handwritten.pdf`) into code. Every number below must match the handwritten pages.

$$\hat{\mathbf y}=X\mathbf m+\mathbf b,\quad X=\begin{bmatrix}1&3\\4&10\end{bmatrix},\ \mathbf y=\begin{bmatrix}5\\6\end{bmatrix},\ \mathbf m=\begin{bmatrix}-1\\2\end{bmatrix},\ \mathbf b=\begin{bmatrix}1\\1\end{bmatrix},\quad J=\frac1n\sum_i(\hat y_i-y_i)^2$$

* **Matrix form throughout** - predictions use `X @ m + b`, the chain-rule gradient uses `X.T @ error`.
* `b` is one shared bias written once per data point, so both entries always stay equal.
* Learning rate α = 0.01 (not given in the brief; stable because α < 2/127.5 ≈ 0.0157).
* Number of updates = 3 = number of group members.

In [1]:
# Setup: only runs in Google Colab, does nothing on a laptop
import os, sys
if "google.colab" in sys.modules:
    REPO = "https://github.com/Abdull-Kudus/ML--Formative--3-Group-20.git"
    if not os.path.exists("/content/repo"):
        os.system(f"git clone -q {REPO} /content/repo")
    os.chdir("/content/repo/notebooks")
    os.system('pip install -q "scipy>=1.15"')

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.differentiate import derivative     # SciPy >= 1.15

IMAGES_DIR = os.path.join("..", "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

X = np.array([[1.0, 3.0], [4.0, 10.0]])
Y = np.array([5.0, 6.0])
M_INITIAL = np.array([-1.0, 2.0])
B_INITIAL = np.array([1.0, 1.0])
LEARNING_RATE = 0.01
N_ITERATIONS = 3


def save_and_show(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, filename), dpi=120)
    plt.show()

## 1. Model and cost function (MSE)

In [3]:
def predict(X, m, b):
    return X @ m + b                    # matrix multiplication, then add the bias vector


def mse(y_hat, y):
    return np.mean((y_hat - y) ** 2)    # J = (1/n) * sum((y_hat - y)^2)


def cost_equation(params, X=X, y=Y):
    # J written as an equation of the three parameters [m1, m2, b].
    # Each parameter may be a NumPy array, because SciPy evaluates several step sizes at once.
    m1, m2, b = params
    return sum((X[i, 0] * m1 + X[i, 1] * m2 + b - y[i]) ** 2 for i in range(len(y))) / len(y)


print("Initial predictions:", predict(X, M_INITIAL, B_INITIAL), " initial J:", mse(predict(X, M_INITIAL, B_INITIAL), Y))

Initial predictions: [ 6. 17.]  initial J: 61.0


## 2. SciPy: a function that accepts an equation and computes its derivative

`derivative_of(equation, point)` passes any one-variable equation to `scipy.differentiate.derivative` (finite differences, refined until the estimate stops changing). `partial_derivatives` calls it once per parameter, holding the others fixed, to build the gradient.

Check on an equation whose answer we know: $f(x)=3x^2+2x \Rightarrow f'(x)=6x+2$, so $f'(2)=14$.

In [4]:
def derivative_of(equation, point):
    return float(derivative(equation, point).df)


def partial_derivatives(equation, params):
    gradient = []
    for k in range(len(params)):
        def along_k(value, k=k):
            moved = list(params)
            moved[k] = value
            return equation(moved)
        gradient.append(derivative_of(along_k, params[k]))
    return np.array(gradient)


print("f'(2) for f(x) = 3x^2 + 2x:", round(derivative_of(lambda x: 3 * x**2 + 2 * x, 2.0), 6))

f'(2) for f(x) = 3x^2 + 2x: 14.0


## 3. Chain rule (for checking SciPy)

$$\frac{\partial J}{\partial \mathbf m}=\underbrace{\frac{\partial J}{\partial \hat{\mathbf y}}}_{\frac2n(\hat{\mathbf y}-\mathbf y)}\cdot\underbrace{\frac{\partial \hat{\mathbf y}}{\partial \mathbf m}}_{X}=\frac2n X^T(\hat{\mathbf y}-\mathbf y),\qquad
\frac{\partial J}{\partial b}=\frac2n\,\mathbf 1^T(\hat{\mathbf y}-\mathbf y)$$

In [5]:
def chain_rule_gradient(X, y, m, b):
    n = len(y)
    error = predict(X, m, b) - y
    return (2.0 / n) * X.T @ error, (2.0 / n) * np.ones(n) @ error

## 4. Gradient descent - every step printed

Each iteration prints: ŷ = Xm + b, the error, the cost, the SciPy gradient **and** the chain-rule gradient (they must match), and the updates of m and b.

In [6]:
def gradient_descent(X, y, m, b, learning_rate=LEARNING_RATE, n_iterations=N_ITERATIONS, verbose=True):
    m, b = m.astype(float).copy(), b.astype(float).copy()
    history = {"m1": [m[0]], "m2": [m[1]], "b": [b[0]], "cost": [mse(predict(X, m, b), y)]}
    for it in range(1, n_iterations + 1):
        y_hat = predict(X, m, b)                                         # step 1: predictions
        error = y_hat - y                                                # step 2: error
        cost = mse(y_hat, y)                                             # step 3: cost
        grad = partial_derivatives(lambda p: cost_equation(p, X, y), [m[0], m[1], b[0]])   # step 4: SciPy
        grad_m, grad_b = grad[:2], grad[2]
        check_m, check_b = chain_rule_gradient(X, y, m, b)
        new_m = m - learning_rate * grad_m                               # step 5: update m
        new_b = b - learning_rate * grad_b                               # step 6: update b (both entries)
        if verbose:
            print(f"--- Iteration {it} ---")
            print(f"  y_hat = X @ m + b = {X.tolist()} @ {np.round(m, 6).tolist()} + {np.round(b, 6).tolist()}"
                  f" = {np.round(y_hat, 6).tolist()}")
            print(f"  error = y_hat - y = {np.round(error, 6).tolist()}    J = {cost:.8f}")
            print(f"  SciPy:      dJ/dm = {np.round(grad_m, 6).tolist()}   dJ/db = {grad_b:.6f}")
            print(f"  Chain rule: dJ/dm = {np.round(check_m, 6).tolist()}   dJ/db = {check_b:.6f}")
            print(f"  m = {np.round(m, 6).tolist()} - {learning_rate} x {np.round(grad_m, 6).tolist()} = {np.round(new_m, 6).tolist()}")
            print(f"  b = {np.round(b, 6).tolist()} - {learning_rate} x ({grad_b:.6f}) = {np.round(new_b, 6).tolist()}\n")
        m, b = new_m, new_b
        history["m1"].append(m[0]); history["m2"].append(m[1]); history["b"].append(b[0])
        history["cost"].append(mse(predict(X, m, b), y))
    return m, b, history


m_final, b_final, history = gradient_descent(X, Y, M_INITIAL, B_INITIAL)

--- Iteration 1 ---
  y_hat = X @ m + b = [[1.0, 3.0], [4.0, 10.0]] @ [-1.0, 2.0] + [1.0, 1.0] = [6.0, 17.0]
  error = y_hat - y = [1.0, 11.0]    J = 61.00000000
  SciPy:      dJ/dm = [45.0, 113.0]   dJ/db = 12.000000
  Chain rule: dJ/dm = [45.0, 113.0]   dJ/db = 12.000000
  m = [-1.0, 2.0] - 0.01 x [45.0, 113.0] = [-1.45, 0.87]
  b = [1.0, 1.0] - 0.01 x (12.000000) = [0.88, 0.88]

--- Iteration 2 ---
  y_hat = X @ m + b = [[1.0, 3.0], [4.0, 10.0]] @ [-1.45, 0.87] + [0.88, 0.88] = [2.04, 3.78]
  error = y_hat - y = [-2.96, -2.22]    J = 6.84500000
  SciPy:      dJ/dm = [-11.84, -31.08]   dJ/db = -5.180000
  Chain rule: dJ/dm = [-11.84, -31.08]   dJ/db = -5.180000
  m = [-1.45, 0.87] - 0.01 x [-11.84, -31.08] = [-1.3316, 1.1808]
  b = [0.88, 0.88] - 0.01 x (-5.180000) = [0.9318, 0.9318]

--- Iteration 3 ---
  y_hat = X @ m + b = [[1.0, 3.0], [4.0, 10.0]] @ [-1.3316, 1.1808] + [0.9318, 0.9318] = [3.1426, 7.4134]
  error = y_hat - y = [-1.8574, 1.4134]    J = 2.72381716
  SciPy:      dJ/d